<a href="https://colab.research.google.com/github/puhspi04/myproject/blob/main/mobilenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [2]:
import zipfile
import os

zip_path = list(uploaded.keys())[0]  # gets uploaded file name
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

Dataset extracted to: /content/dataset


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNet
from tensorflow.keras import layers, models

In [4]:
train_dir = "/content/dataset/train"
val_dir = "/content/dataset/val"

In [7]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    print(root)

/content/dataset
/content/dataset/Main Dataset
/content/dataset/Main Dataset/UTENSIL
/content/dataset/Main Dataset/JUICE_BOX
/content/dataset/Main Dataset/BOTTLE
/content/dataset/Main Dataset/STYROFOAM
/content/dataset/Main Dataset/CAN
/content/dataset/Main Dataset/MILK_CARTON


In [8]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "/content/dataset/Main Dataset",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/content/dataset/Main Dataset",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 1248 images belonging to 6 classes.
Found 310 images belonging to 6 classes.


In [9]:
base_model = MobileNet(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
for layer in base_model.layers:
    layer.trainable = False

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
epochs = 20

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs
)

Epoch 1/20
18/39 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.5795 - loss: 1.2734

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


39/39 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.8750 - loss: 0.4269 - val_accuracy: 0.9290 - val_loss: 0.2134
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.9840 - loss: 0.0590 - val_accuracy: 0.9516 - val_loss: 0.1627
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.9944 - loss: 0.0266 - val_accuracy: 0.9516 - val_loss: 0.1639
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.9992 - loss: 0.0101 - val_accuracy: 0.9516 - val_loss: 0.1655
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 72s 2s/step - accuracy: 1.0000 - loss: 0.0055 - val_accuracy: 0.9516 - val_loss: 0.1602
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 1.0000 - loss: 0.0038 - val_accuracy: 0.9581 - val_loss: 0.1482
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 1.0000 - loss: 0.0030 - val_accuracy: 0.9581 - val_loss: 0.1472
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 0.9613 - val_loss: 0.1464
Epo

In [13]:
model.save("mobilenet_model.h5")

In [14]:
print(f"Final Training Loss: {history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.4f}")

Final Training Loss: 0.0005
Final Validation Loss: 0.1520


In [15]:
print("Final Training Accuracy:", history.history['accuracy'][-1])
print("Final Validation Accuracy:", history.history['val_accuracy'][-1])

Final Training Accuracy: 1.0
Final Validation Accuracy: 0.9580644965171814


In [17]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.3,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7,1.3]
)

In [18]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

In [29]:
from google.colab import files
uploaded = files.upload()

Saving Screenshot 2026-03-24 004208.png to Screenshot 2026-03-24 004208.png


In [30]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = list(uploaded.keys())[0]

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = img_array / 255.0   # normalize
img_array = np.expand_dims(img_array, axis=0)  # make batch

In [31]:
prediction = model.predict(img_array)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step


In [32]:
class_labels = list(train_data.class_indices.keys())

predicted_class = class_labels[np.argmax(prediction)]

print("Predicted class:", predicted_class)

Predicted class: CAN
